# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: CTR / Engagement Opportunity Scoring** (one of the four predefined lanes).

Why this one: it's the lane that maps most directly onto a decision a client-facing team would
actually act on. Ranking Signal Analysis is more open-ended correlation work with a fuzzier
"done" state; Structured Content Archetype Clustering has no real target or metric to hold
myself to; Refresh/Content Opportunity Scoring is already solved end-to-end by the shipped
pipeline in this repo, which leaves me extending someone else's answer rather than building my
own. CTR/Engagement scoring has a clear observed target (`ctr`), a clear population to check it
against (pages already ranking reasonably well), and — as the numbers below show — a real,
sizeable gap worth explaining.


In [1]:
lane = "CTR / Engagement Opportunity Scoring"
print(f"Provisional lane: {lane}")
print("(confirmed via docs/ml-intern-dataset-and-lane-guide.md, section: Which Lanes Are Ready)")


Provisional lane: CTR / Engagement Opportunity Scoring
(confirmed via docs/ml-intern-dataset-and-lane-guide.md, section: Which Lanes Are Ready)


## 2. The question: decision, action, cost of a wrong call

**Research question:** Among pages that already rank reasonably well (page 1 or the "striking
distance" zone), which ones are getting far fewer clicks than similar pages at the same
position — and can that gap be explained by fixable page-level factors rather than just noise?

**Decision this improves:** which already-well-ranked pages an editor should open first to fix
the *listing itself* (title, meta description, content framing) rather than chase more ranking
gains. Ranking work and CTR work are different jobs; right now nothing in this dataset
distinguishes them.

**Who acts, and how:** a content editor or SEO strategist (at an agency like the one I run, this
would be me or a client-facing strategist) works a ranked queue — top of the list first — and
rewrites the on-page listing elements for that page.

**Cost of a wrong call:** two different wrong calls, and they cost differently.
- **False positive** (flagging a page as underperforming when it isn't really fixable): wastes an
  editor's time rewriting something that wasn't broken — a few hours per page, recoverable.
- **False negative** (missing a page that's genuinely leaking clicks): the page keeps ranking well
  and keeps losing clicks nobody notices — the more expensive miss, since it's invisible unless
  someone goes looking, which is exactly what this lane is for.

Because a missed opportunity is quieter and more expensive than a wasted edit, I'd rather this
lane err toward flagging more candidates and let a human do a final gut-check, than stay silent
and let real gaps go unnoticed.


In [2]:
decision = "Which already-well-ranked pages should an editor fix first for CTR, not rank?"
actor = "Content editor / SEO strategist"
action = "Rewrite title + meta description + listing framing for the flagged page"
cost_of_false_positive = "Wasted editor hours on a page that wasn't actually broken"
cost_of_false_negative = "A real, ongoing click leak that stays invisible"

for k, v in {
    "decision": decision,
    "actor": actor,
    "action": action,
    "cost_of_false_positive": cost_of_false_positive,
    "cost_of_false_negative": cost_of_false_negative,
}.items():
    print(f"{k}: {v}")


decision: Which already-well-ranked pages should an editor fix first for CTR, not rank?
actor: Content editor / SEO strategist
action: Rewrite title + meta description + listing framing for the flagged page
cost_of_false_positive: Wasted editor hours on a page that wasn't actually broken
cost_of_false_negative: A real, ongoing click leak that stays invisible


## 3. Quick look at the data (2-3 real numbers)

I looked only at pages that already rank on page 1 or in striking distance
(`position_tier` in `top_3`, `page_1`, `striking`) with at least 100 impressions in the trailing
90 days, so the CTR numbers aren't just noise from a handful of impressions. Numbers below are
computed directly from `data/raw/content_refresh_anonymized.csv` — not estimated.


In [3]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df = df[df["avg_position"] > 0]  # avg_position == 0 means "no data", not rank zero

good_pos = df[df["position_tier"].isin(["top_3", "page_1", "striking"])]
print(f"1) Pages already ranking page-1-or-better: {len(good_pos):,} of {len(df):,} total")

vol = good_pos[good_pos["impressions_90d"] >= 100]
tier_median_ctr = vol.groupby("position_tier")["ctr"].transform("median")
under = vol[vol["ctr"] < tier_median_ctr]

print(f"2) Of those with >=100 impressions, {len(under):,} of {len(vol):,} "
      f"({len(under) / len(vol):.0%}) sit BELOW their own position tier's median CTR")

print(f"3) Those underperforming pages carried {int(under['impressions_90d'].sum()):,} "
      f"impressions but only {int(under['clicks_90d'].sum()):,} clicks in 90 days, "
      f"spread across {under['client_id'].nunique()} of {df['client_id'].nunique()} clients")


1) Pages already ranking page-1-or-better: 20,234 of 28,795 total
2) Of those with >=100 impressions, 7,393 of 15,069 (49%) sit BELOW their own position tier's median CTR
3) Those underperforming pages carried 47,886,454 impressions but only 55,271 clicks in 90 days, spread across 27 of 31 clients


## 4. Careful words: what I can and can't claim

**What I can claim, once I've done the work:** which currently well-ranked pages have a CTR gap
relative to comparable pages (same position tier, and later, same intent/content type) — an
**observed, decision-support** flag, not a certainty. I can say "this page underperforms its
peers by X points" because that's measured directly from the data.

**What I can never claim:** that fixing the listing *will* recover the gap (that's a causal claim
I have no experiment to back — I only have observational data, no before/after test), or that
this model "predicts Google" or explains why the algorithm ranks the way it does. The gap is
real; the reason for the gap and the effect of any fix are hypotheses an editor tests, not facts
this notebook proves.

**Language I'll use in this lane going forward:** "observed gap," "directional signal,"
"decision-support flag," "candidate for review" — never "proven," "will improve," or "causes."


In [4]:
# No computation needed here -- this cell exists to keep the pattern consistent
# (markdown thinking backed by a code cell), per the skeleton's own note.
careful_words = ["observed", "directional", "decision-support", "candidate for review"]
avoid_words = ["proven", "will improve", "causes", "predicts Google"]
print("Use:", ", ".join(careful_words))
print("Avoid:", ", ".join(avoid_words))


Use: observed, directional, decision-support, candidate for review
Avoid: proven, will improve, causes, predicts Google


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.